In [ ]:
import json
import pandas as pd
import glob
from math import ceil, sqrt
import matplotlib.pyplot as plt
import os
import sys
import numpy as np
import seaborn as sns
import traceback
import plotly.graph_objects as go
import plotly.express as px
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from tqdm.notebook import trange, tqdm
import re
import oapackage
from sklearn.model_selection import ParameterGrid
import copy
import plotly.figure_factory as ff
import datapane as dp
from colour import Color
from decimal import Decimal
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.precision', 5)
figSize = (20,  20)

baseFolder = 'C:/Users/Sergi/Documents/networkExperiments/output/'

# True: ascending, False: descending
metricsSorting = {'numberOfMigrations': True, 'energy': True, 'slaAverage': True, 'averagePercentageOfNonSatisfiedByTotalMips': True,
                'networkPhysicalResults-totalBWUsed': True, 'networkPhysicalResults-totalRAMInBW': True,
                'numberOfHostShutdowns': True, 'slaTimePerActiveHost': True, 'totalSaturatedHosts': True, 'elapsedTime': True,
                'vmTotalAllocatedMips': False, "averagePercentageOfNonSatisfiedMips": True,
                 "networkPhysicalResults-totalSaturatedData": True, "sla": True, 'averageCPUPercAllocatedOfVms': False,
                 'averageCPUOverUsageNonAllocatedOfOverUsedVM': False, "totalWastedMIPS": True, "avPercUnsatVMs": True,
                  "totalUnsatisfiedVMs": True, "slaOverall": True, "totalNonSatisfiedMips": True,
                  "totalUsedHosts": True, "totalUnsatisfiedHosts": True, "energyMigrations": True, "energyMigrationsPaths": True,
                  "energyWithExtraHost": True, "energy+hosts+migrations": True
                 }

goodColor = 'background-color: lightgreen'
neutralColor = 'background-color: khaki'
badColor = 'background-color: lightcoral'
whiteColor = 'background-color: white'

experimentConfigurations = []
experimentConfigurationsWithLists = set(["vmSignalProcessingParamsBasic", "hostsDistribution",
                               "hostSignalProcessingParams", "vmForecastingTechniqueParamsBasic",
                               "limitAdjustmentParams", "vmsDistribution", "signalProcessingParams",
                               "techniquesConfiguration", "interactions", "forecastingTechniqueParams",
                               "hostForecastingTechniqueParams", "interactionsDistribution",
                               "vmSignalProcessingParamsAccurate", "vmForecastingTechniqueParamsAccurate"])

def getStatisticsDataframe(currentFolder):
    files = glob.glob(currentFolder+'/*_data.json')
    dictDataList = []
    for file in tqdm(files):
        with open(file) as fd:
            json_data = json.load(fd)
            dictData = {}
            for key in json_data.keys():
                for key2 in json_data[key]:
                    value = json_data[key][key2]                    
                    if(isinstance(value, (int, float, str, bool))):
                        if key == "networkResults" or key == "networkPhysicalResults":
                            dictData[key+"-"+key2] = value
                        else:
                            dictData[key2] = value
                    if key2 == "limitAdjustmentParams":
                        for index, val in enumerate(value):
                            dictData[key2+"-"+str(index)] = val
                    if key2 == "slaMetrics":
                        for keySla in value.keys():
                            dictData[key2+"-"+keySla] = value[keySla]
                    if key2 == "usedHostsPerSnapshot":
                        currentSer = pd.Series(value)
                        dictData["usedHostsPerSnapshot-mean"] = currentSer.mean()
                        dictData["usedHostsPerSnapshot-std"] = currentSer.std()
                        dictData["usedHostsPerSnapshot-median"] = currentSer.median()
                    if key2 in experimentConfigurationsWithLists:
                        dictData[key2] = value
                    if key2 == "totalFreeMIPSFromActiveHostsPerSnapshot":
                        dictData["totalFreeMIPSFromActiveHostsPerSnapshot-sum"] = pd.Series(value).sum()
                    
                    
                    if key == "networkPhysicalResults" and key2 == "totalMigratedRAM":
                        #dictData["energyMigrations"] = value * (1.7892 + 1.7596) # CLoudESE
                        dictData["energyMigrations"] = 4.096 * value + 20.165 # Seeking for the Optimal Energy Modelisation Accuracy to Allow Efficient Datacenter Optimizations
                        dictData["energyMigrations"] *= 1.1 #### MOD REVIEW PAPER 3
                    if key == "networkPhysicalResults" and key2 == "totalRAMInBW":
                        #dictData["energyMigrationsPaths"] = value * (1.7892 + 1.7596) # CLoudESE
                        dictData["energyMigrationsPaths"] = 4.096 * value + 20.165 # Seeking for the Optimal Energy Modelisation Accuracy to Allow Efficient Datacenter Optimizations
                    if key == "networkPhysicalResults" and key2 == "totalBWUsed":
                        dictData["energyVMCommunications"] = 4.096 * value + 20.165
                            
            if dictData["techniqueLoader"] == "v2" and dictData["hostSignalProcessing"] == "none":
                dictData["hostSignalProcessing"] = dictData["signalProcessing"]
                dictData["hostForecastingTechnique"] = dictData["forecastingTechnique"]
                dictData["vmSignalProcessingAccurate"] = dictData["signalProcessing"]
                dictData["vmForecastingTechniqueAccurate"] = dictData["forecastingTechnique"]
                dictData["vmSignalProcessingBasic"] = dictData["signalProcessing"]
                dictData["vmForecastingTechniqueBasic"] = dictData["forecastingTechnique"]
            dictData["elapsedTime"] = json_data["timeResults"]["endTime"] - json_data["timeResults"]["startTime"]
            
            dictData["energyMigrations"] = dictData["energyMigrations"] / (3600 * 1000)
            dictData["energyMigrationsPaths"] = dictData["energyMigrationsPaths"] / (3600 * 1000)
            dictData["energyVMCommunications"] = dictData["energyVMCommunications"] / (3600 * 1000)
            
            if "energyWithExtraHost" in dictData:
                dictData["energy+hosts+migrations"] = dictData["energyWithExtraHost"] + dictData["energyMigrations"]
                dictData["energy+hosts+migrations_paths"] = dictData["energyWithExtraHost"] + dictData["energyMigrationsPaths"]
                dictData["energy+hosts+migrations+vm_comm"] = dictData["energyWithExtraHost"] + dictData["energyMigrations"] + dictData["energyVMCommunications"]
                dictData["energy+hosts+migrations_paths+vm_comm"] = dictData["energyWithExtraHost"] + dictData["energyMigrationsPaths"] + dictData["energyVMCommunications"]

            dictDataList.append(dictData)
    df = pd.DataFrame(dictDataList)
    df.fillna(-1, inplace=True)
    return df
   
def loadExperimentConfigurations():
    files = glob.glob(currentFolder+'/*_data.json')
    with open(files[0]) as fd:
        json_data = json.load(fd)
        experimentConfigurations.clear()
        experimentConfigurations.extend(json_data["experimentConfiguration"].keys())

def showStatistics(df, metrics, characteristics):
    #fig, axs = plt.subplots(len(metrics), 1, figsize=(15,  13*len(metrics)))
    for metric in metrics:
        fig, ax = plt.subplots(1, 1, figsize=figSize)
        #df.boxplot(column=metric, by=characteristics, ax=ax, rot=90, showfliers=False)
        #color = {"boxes": "g", "whiskers": "o", "medians": "b", "caps": "r"}
        df.boxplot(column=metric, by=characteristics, ax=ax, showfliers=False, vert=False, color={'medians': 'blue'}, 
        medianprops={'linestyle': '-', 'linewidth': 5})
        plt.show()
        
def generateTable(df, metrics, technique_renames={}, metrics_renames={}, override=False,
                  ref_index=0, values_with_perc = False, require_times_100={}, num_decimals={}, filename="table_output", to_excel=False, set_index=True):
    if set_index:
        df = df.set_index("id")
    currentTable = df[metrics]
    for r in require_times_100:
        currentTable[r] *= 100
    finalTable = currentTable.T
    finalTable.rename(technique_renames, axis=1, inplace=True)
    #print(finalTable.columns)
    for col in finalTable.columns[:]:
    #    finalTable[col+"_%_com_ref"] = (1.0 - (finalTable[col] / finalTable[finalTable.columns[ref_index]])) * 100 * -1
    #    finalTable[col+"_%_ref_to_oth"] = (1.0 - (finalTable[finalTable.columns[ref_index]] / finalTable[col])) * 100
        finalTable[col] = finalTable.apply(lambda row: ("{:."+str(num_decimals.get(row.name, 2))+"f}").format(row[col]), axis=1)
        finalTable[col] = finalTable.apply(lambda row: round(float(row[col]),0) if float(row[col]) > 1000.0 else row[col].rstrip("0").rstrip(".") if "." in row[col] else row[col], axis=1)
    #    finalTable[col+"_final"] = finalTable.apply(lambda row: ("{:."+str(num_decimals.get(row.name, 2))+"f}").format(row[col]) + " (" + ("{:.2f}%").format(row[col+"_%_com_ref"]) + ")", axis=1)
    #    if values_with_perc:
    #        finalTable.drop([col, col+"_%_com_ref", col+"_%_ref_to_oth"], axis=1, inplace=True)
    #        finalTable.rename({col+"_final": col}, axis=1, inplace=True)
    finalTable.rename(metrics_renames, axis=0, inplace=True)
    display(finalTable)
    if to_excel:
        finalTable.to_excel(filename+".xlsx")
    return finalTable

def generateTable_bak(df, metrics, technique_renames={}, metrics_renames={}, override=False,
                  ref_index=0, values_with_perc = False, require_times_100={}, num_decimals={}, filename="table_output", to_excel=False, set_index=True):
    if set_index:
        df = df.set_index("id")
    currentTable = df[metrics]
    for r in require_times_100:
        currentTable[r] *= 100
    finalTable = currentTable.T
    finalTable.rename(technique_renames, axis=1, inplace=True)
    for col in finalTable.columns[1:]:
        finalTable[col+"_%_com_ref"] = (1.0 - (finalTable[col] / finalTable[finalTable.columns[ref_index]])) * 100 * -1
        finalTable[col+"_%_ref_to_oth"] = (1.0 - (finalTable[finalTable.columns[ref_index]] / finalTable[col])) * 100
        decimals = num_decimals.get(col, 2)
        finalTable[col+"_final"] = finalTable.apply(lambda row: ("{:."+str(num_decimals.get(row.name, 2))+"f}").format(row[col]) + " (" + ("{:.2f}%").format(row[col+"_%_com_ref"]) + ")", axis=1)
        if values_with_perc:
            finalTable.drop([col, col+"_%_com_ref", col+"_%_ref_to_oth"], axis=1, inplace=True)
            finalTable.rename({col+"_final": col}, axis=1, inplace=True)
    finalTable.rename(metrics_renames, axis=0, inplace=True)
    display(finalTable)
    if to_excel:
        finalTable.to_excel(filename+".xlsx")
    return finalTable
    
#def showStatistics(df, metrics, characteristics):
#    fig, axs = plt.subplots(len(metrics), 1, figsize=(15,  13*len(metrics)))
#    boxplot = df.boxplot(column=metrics, by=characteristics, ax=axs, rot=90, showfliers=False)
#    plt.show()

def showPlotlyStatistics(df, metrics, characteristic, color=None, hover_data=None, size=None, show=True):
    figures = []
    for metric in metrics:
        title = re.sub(r'(?<!^)(?=[A-Z])', ' ', metric).capitalize()
        fig = px.box(df, x=characteristic, y=metric, color=color, hover_data=hover_data, title=title, template="plotly_white")
        fig.update_layout(title_x=0.5)
        fig.update_layout(
            xaxis = go.layout.XAxis(
                tickangle = 30)
        )
        #fig.update_traces(boxpoints=False) 
        if size is not None:
            fig.update_layout(autosize=False, width=size[0],height=size[1])
        if show:
            fig.show()
        figures.append(fig)
    return figures

def showPlotlyBarStatistics(df, metrics, characteristic, color=None, hover_data=None, ref=None, size=None, show=True, ref_perc=None):
    figures = []
    for metric in metrics:
        title = re.sub(r'(?<!^)(?=[A-Z])', ' ', metric).capitalize()
        dfSelection = None
        if ref is not None:
            dfSelection = df.copy()
            dfSelection[metric] = (1.0 - (dfSelection[metric] / ref.iloc[0][metric])) * 100
            title = title + " (% difference)"
        elif ref_perc is not None:
            dfSelection = df.copy()
            dfSelection[metric + "_perc"] = dfSelection[metric].apply(lambda x: '{:.2f}'.format(x / ref_perc.iloc[0][metric] * 100).rstrip('0').rstrip('.') + "%")
        else:
            dfSelection = df
        if metric == "energyPlus": metric = ["energy", "energyMigrations", "energyOpenCloseHosts"]
        if ref_perc is not None:
            text = metric + "_perc"
        else:
            text = None
        fig = px.bar(dfSelection, x=characteristic, y=metric, color=color,
                     hover_data=hover_data, title=title, template="plotly_white", text=text)
        fig.update_layout(title_x=0.5)
        fig.update_traces(textposition='outside', cliponaxis=False)
        fig.update_layout(
            xaxis = go.layout.XAxis(
                tickangle = 30)
        )
        if size is not None:
            fig.update_layout(autosize=False, width=size[0],height=size[1])
        if show:
            fig.show()
        figures.append(fig)
    return figures

def showPlotlyBarStatisticsWithTextures(df, metrics, characteristic, color=None, hover_data=None, ref=None, size=None, show=True,
                                       pattern_shape=None, pattern_shape_sequence=None):
    figures = []
    for metric in metrics:
        title = re.sub(r'(?<!^)(?=[A-Z])', ' ', metric).capitalize()
        dfSelection = None
        if ref is not None:
            dfSelection = df.copy()
            dfSelection[metric] = (1.0 - (dfSelection[metric] / ref.iloc[0][metric])) * 100
            title = title + " (% difference)"
        else:
            dfSelection = df
        if metric == "energyPlus": metric = ["energy", "energyMigrations", "energyOpenCloseHosts"]
        fig = px.bar(dfSelection, x=characteristic, y=metric, color=color,
                     hover_data=hover_data, title=title, template="plotly_white",
                     pattern_shape=pattern_shape, pattern_shape_sequence=pattern_shape_sequence)
        fig.update_layout(title_x=0.5)
        fig.update_layout(
            xaxis = go.layout.XAxis(
                tickangle = 30)
        )
        if size is not None:
            fig.update_layout(autosize=False, width=size[0],height=size[1])
        if show:
            fig.show()
        figures.append(fig)
    return figures

def getMaxSLALimitedByEnergy(currentDf, ref):
    underEnergyDf = currentDf[currentDf["energy+hosts+migrations"] < ref["energy+hosts+migrations"].iloc[0]]
    minSLA = underEnergyDf[underEnergyDf["slaOverall"] == underEnergyDf["slaOverall"].min()]
    #print(minSLA[["alpha", "window", "energy+hosts+migrations", "slaOverallComp"]])
    return minSLA
    
def getMinEnergyLimitedBySLA(currentDf, ref):
    underSLADf = currentDf[currentDf["slaOverall"] < ref["slaOverall"].iloc[0]]
    minEnergy = underSLADf[underSLADf["energy+hosts+migrations"] == underSLADf["energy+hosts+migrations"].min()]
    #print(minEnergy[["alpha", "window", "energy+hosts+migrations", "slaOverallComp"]])
    return minEnergy
    
def getOptimizedEnergy(currentDf):
    windowValues = currentDf["window"].unique().tolist()
    minEnergies = []
    for windowValue in windowValues:
        currentWindowDf = currentDf[currentDf["window"]==windowValue]
        minEnergyCurrentWindow = currentWindowDf[currentWindowDf["energy+hosts+migrations"] == currentWindowDf["energy+hosts+migrations"].min()]
        #print(minEnergyCurrentWindow[["alpha", "window", "energy+hosts+migrations", "slaOverallComp"]])
        minEnergies.append(minEnergyCurrentWindow)
    finalDf = pd.concat(minEnergies)
    #finalDf.plot.bar(x="window", y="energy+hosts+migrations")
    #plt.show()
    #finalDf.plot.bar(x="window", y="ratioSE")
    #plt.show()
    #finalDf.plot.bar(x="window", y="slaOverallComp", ylim=(0.93, 0.98))
    #plt.show()
    minEnergy = finalDf[finalDf["energy+hosts+migrations"] == finalDf["energy+hosts+migrations"].min()]
    #print(minEnergy[["alpha", "window", "energy+hosts+migrations", "slaOverallComp"]])
    return minEnergy
        
def showPlotlyHeatmapStatistics(df, metrics, characteristic, heatmapColumns, color=None, hover_data=None, ref=None, size=None):
    for metric in metrics:
        heatmapData = df.pivot(heatmapColumns[0], heatmapColumns[1], metric)
        fig = None
        if ref is not None:
            heatmapData = (1.0 - (heatmapData / ref.iloc[0][metric])) * 100
            fig = go.Figure(data=go.Heatmap(
                           z=heatmapData.values,
                           x=heatmapData.columns,
                           y=heatmapData.index,
                           colorscale='RdYlGn',
                            #zmin=-100,    
                            #zmax=100,
                            ))
        else:
            fig = go.Figure(data=go.Heatmap(
                               z=heatmapData.values,
                               x=heatmapData.columns,
                               y=heatmapData.index
                            ))
        fig.layout.xaxis.type = 'category'
        fig.layout.yaxis.type = 'category'
        fig.update_layout(
            title = re.sub(r'(?<!^)(?=[A-Z])', ' ', metric).capitalize(),
            xaxis_title=heatmapColumns[1],
            yaxis_title=heatmapColumns[0],
        )
        if size is not None:
            fig.update_layout(autosize=False, width=size[0],height=size[1])
        fig.show()

def showPlotlySlicedLineSurface(df, metrics, heatmapColumns, colors=["#b2d8ff", "#00264c"], hover_data=None, ref=None, comparative=True, highlights=[], size=None, dtick=None, markerSize=5):
    figures = []
    start_color, end_color = colors
    # list of "N" colors between "start_color" and "end_color"
    numLines = len(df[heatmapColumns[0]].unique())
    colorscale = [x.hex for x in list(Color(start_color).range_to(Color(end_color), numLines))]

    for metric in metrics:
        dfSelection = None
        if ref is not None and comparative is True:
            dfSelection = df[[metric, heatmapColumns[0], heatmapColumns[1]]].copy()
            for value, currentRef in ref.items():
                # TODO: Check correctness
                dfSelection.loc[dfSelection[heatmapColumns[0]]==value, metric] = (1.0 - (dfSelection.loc[dfSelection[heatmapColumns[0]]==value, metric] / currentRef.iloc[0][metric])) * 100
                #(1.0 - (dfSelection.loc[dfSelection[heatmapColumns[0]]==value][metric] / currentRef[heatmapColumns[0]].iloc[0][metric])) * 100
            #dfSelection[metric] = (1.0 - (dfSelection[metric] / ref[heatmapColumns[0]].iloc[0][metric])) * 100
            #dfSelection[metric] = (1.0 - (dfSelection[metric] / ref.iloc[0][metric])) * 100
        else:
            dfSelection = df
        fig = px.line(dfSelection, x=heatmapColumns[1], y=metric,
                      color=heatmapColumns[0], template="plotly_white", hover_data=hover_data)
        for i in range(len(fig.data)):
            fig.data[i].line.color = colorscale[i]
            fig.data[i].update(mode='markers+lines', marker_symbol=i, marker_size=markerSize)
        
        for highlight in highlights:
            fig.data[highlight["index"]].line.color = highlight["color"]
            fig.data[highlight["index"]].line.width = 3
            
        if ref is not None and comparative is False:
            maxX = dfSelection[heatmapColumns[1]].values.max()
            minX = dfSelection[heatmapColumns[1]].values.min()
            refColors = px.colors.qualitative.Light24
            i=0
            for annotation, currentRef in ref.items():
                #fig.add_hline(y=currentRef.iloc[0][metric])
                currentY = currentRef.iloc[0][metric]
                fig.add_trace(go.Scatter(x=[minX, maxX], 
                         y=[currentY, currentY], 
                         mode='lines', 
                         name=annotation,
                         line=dict(dash='dash', color=refColors[i])))
                i+=1
                
        title = re.sub(r'(?<!^)(?=[A-Z])', ' ', metric).capitalize()
        fig.update_layout(
            title = title if not comparative else title + " (% difference)",
            xaxis_title=heatmapColumns[1],
            yaxis_title=metric,
        )
        if size is not None:
            fig.update_layout(autosize=False, width=size[0],height=size[1])
        if dtick is not None:
            fig.layout.xaxis.dtick = dtick
        fig.show()
        figures.append(fig)
    return figures

def showPlotly3DSurfaceStatistics(df, metrics, characteristic, heatmapColumns, color=None, hover_data=None, ref=None, size=None):
    figures = []
    for metric in metrics:
        heatmapData = df.pivot(heatmapColumns[0], heatmapColumns[1], metric)
        if ref is not None:
            heatmapData = (1.0 - (heatmapData / ref.iloc[0][metric])) * 100
        fig = go.Figure(data=[go.Surface(
                    z=heatmapData.values,
                    x=heatmapData.columns,
                    y=heatmapData.index)
                ],
                layout={"scene":
                            {"camera": {"projection": {"type": "orthographic"}},
                             "xaxis": dict(title=heatmapColumns[1]),
                             "yaxis": dict(title=heatmapColumns[0]),
                             "zaxis": {"title": metric}
                           }
                       }
               )
        fig.update_traces(contours_z=dict(show=True, usecolormap=True, project_z=True))
        fig.layout.xaxis.type = 'category'
        fig.layout.yaxis.type = 'category'
        fig.update_layout(
            title = re.sub(r'(?<!^)(?=[A-Z])', ' ', metric).capitalize(),
            xaxis_title=heatmapColumns[1],
            yaxis_title=heatmapColumns[0],
        )
        if size is not None:
            fig.update_layout(autosize=False, width=size[0],height=size[1])
        fig.show()
        figures.append(fig)
    return figures

def showPlotly3DMultipleSurfaceStatistics(dfDict, metrics, characteristic, heatmapColumns, color=None, hover_data=None, ref=None, size=None):
    figures = []
    lighting_effects = dict(ambient=0.4, diffuse=0.5, roughness = 0.9, specular=0.6, fresnel=0.2)
    for metric in metrics:
        data = []
        minValue = None
        maxValue = None
        if ref is None:
            minValue  = min([min(dfDict[key][metric]) for key in dfDict.keys()])
            maxValue  = max([max(dfDict[key][metric]) for key in dfDict.keys()])
        for technique, df in dfDict.items():
            heatmapData = df.pivot(heatmapColumns[0], heatmapColumns[1], metric)
            if ref is not None:
                heatmapData = (1.0 - (heatmapData / ref[technique].iloc[0][metric])) * 100
                data.append(go.Surface(
                            z=heatmapData.values,
                            x=heatmapData.columns,
                            y=heatmapData.index,
                            surfacecolor=heatmapData.values,
                            name=technique,
                            cmin=-100,    
                            cmax=100,
                            colorscale='RdYlGn',
                            colorbar=dict(lenmode='fraction', len=0.75),
                            contours_z=dict(show=True, usecolormap=True, project_z=False),
                            opacity=0.5,
                            lighting = lighting_effects
                            ))
                data.append((go.Scatter3d(x=[0], y=[0], z=[0],
                            name=technique, mode='markers')))
            else:
                data.append(go.Surface(
                            z=heatmapData.values,
                            x=heatmapData.columns,
                            y=heatmapData.index,
                            surfacecolor=heatmapData.values,
                            name=technique,
                            cmin=minValue,    
                            cmax=maxValue,
                            colorbar=dict(lenmode='fraction', len=0.75),
                            contours_z=dict(show=True, usecolormap=True, project_z=False, start=minValue, end=maxValue),
                            opacity=0.5,
                            lighting = lighting_effects
                            ))
                data.append((go.Scatter3d(x=[0], y=[0], z=[heatmapData[0][0]],
                            name=technique, mode='markers')))
        if ref is not None:
            heatmapData = dfDict[list(dfDict.keys())[0]].pivot(heatmapColumns[0], heatmapColumns[1], metric)
            heatmapData[:] = 0
            data.append(go.Surface(
                            z=heatmapData.values,
                            x=heatmapData.columns,
                            y=heatmapData.index,
                            opacity=0.5,
                            surfacecolor=heatmapData.values,
                            name="z0",
                            cmin=-100,
                            cmax=100,
                            colorbar=dict(lenmode='fraction', len=0.75),
                            colorscale='RdYlGn',
                            ))
        fig = go.Figure(data=data,
                layout={"scene":
                            {"camera": {"projection": {"type": "orthographic"}},
                             "xaxis": dict(title=heatmapColumns[1]),
                             "yaxis": dict(title=heatmapColumns[0]),
                             "zaxis": {"title": metric, "nticks": len(dfDict)*5}
                           }
                       }
               )
        #fig.update_traces(contours_z=dict(show=True, usecolormap=True, project_z=True))
        
        fig.layout.xaxis.type = 'category'
        fig.layout.yaxis.type = 'category'
        fig.update_layout(
            title = re.sub(r'(?<!^)(?=[A-Z])', ' ', metric).capitalize(),
            xaxis_title=heatmapColumns[1],
            yaxis_title=heatmapColumns[0],
            legend_x=-0.15,
        )
        if size is not None:
            fig.update_layout(autosize=False, width=size[0], height=size[1])
        figures.append(fig)
        fig.show()
    return figures
        
def showDotPlot(df, x, y, color):
    fig = px.scatter(df,
                     x=x,
                     y=y,
                     color=color)
    fig.show()
    
def getAllowedFilteredDf(df, filters):
    filteredDf = df.copy()
    for k,v in filters.items():
        filteredDf = filteredDf[filteredDf[k].isin(v)]
    return filteredDf

def getAllExceptFilteredDf(df, filters):
    filteredDf = df.copy()
    for k,v in filters.items():
        filteredDf = filteredDf[~filteredDf[k].isin(v)]
    return filteredDf

def showPlotlyScatter(df, x, y, color=None, hover_data=[], size=None):
    fig = px.scatter(df, x=x, y=y, color=color, hover_data=hover_data, size=size)
    fig.show()

def showSeabornScatter(df, x, y, style=None, hue=None):
    plt.figure(figsize=[15,7])
    sns.scatterplot(data=df, x=x, y=y, style=style, hue=hue)
    
options = ["experimentOutputFolder", "vmOptimizerPolicy", "selectionPolicy", "selectionMode",
              "hostOverSaturationPolicy", "hostUnderUtilisationPolicy", "vmMigrationPolicy",
              "allocationPolicy", "techniqueLoader", "dataSource", "signalProcessing", "forecastingTechnique",
              "hostSignalProcessing", "hostForecastingTechnique", "vmSignalProcessingAccurate", "vmForecastingTechniqueAccurate",
              "vmSignalProcessingBasic", "vmForecastingTechniqueBasic", "limitAdjustment", "defaultHistoryLength",
              "forecastingHistoryLength", "topology", "hosts", "vms", "workload", "workloadTrace", "mipsStatsInterpolation", "parameter",
              "numCloudlets", "utilizationThreshold", "randomSeed", "migrationInterval", "dataStartStep",
              "simulationTimeLimit", "internalClusterRate", "externalClusterRate", "printer", "enableOutput",
              "outputToFile", "hostAllocationUtilization", "onlyInitialHosts", "outputFullTrace", "outputCSVTrace",
              "outputGraphTrace", "overrideLinkWeight", "logicLinkWeight", "defaultWeight", "defaultWeightInterconnected",
              "isTunning", "tunningValue", "window", "alpha", "hostForecastingResume", "vmForecastingResumeAccurate",
              "vmForecastingResumeBasic", "network"]

def showOptions(df, showMoreThanOne = False):
    print("Options:")
    for option in options:
        if option in df.columns:
            valueCounts = df[option].value_counts()
            if(len(valueCounts) > 1):
                print(option+":")
                print("\t", end="")
                for index, value in valueCounts.items():
                    print(f" {index}[{value}]", end="")
                print("")
        else:
            print("Option not found:", option)
    print("End options")
            
def getRankingAndPercentiles(df, metrics):
    scaler = MinMaxScaler()
    rankDf = df.copy()
    scoreMetrics = [[metric, metricsSorting[metric]] for metric in metrics]
    for metric, ascendingSorting  in scoreMetrics:
        rankDf[metric+"-rank"] = rankDf[metric].rank(ascending=not ascendingSorting, pct=True)
        rankDf[metric+"-perc"] = scaler.fit_transform(rankDf[metric].values.reshape(-1,1))
        if not ascendingSorting:
            rankDf[metric+"-perc"] = 1.0 - rankDf[metric+"-perc"]

    metricsRank = [m + "-rank" for m in metrics]
    metricsPerc = [m + "-perc" for m in metrics]

    rankDf["rank"] = rankDf[metricsRank].sum(axis=1)
    rankDf["perc"] = rankDf[metricsPerc].sum(axis=1)
    rankDf["final-rank"] = rankDf["rank"].rank(ascending=True, pct=False)
    rankDf["final-perc"] = rankDf["perc"].rank(ascending=True, pct=False)

    return rankDf

def showPareto2DBasic(df, metricX, metricY):
    paretoData = df.copy()
    paretoData["front"] = False
    pareto=oapackage.ParetoDoubleLong()
    x = metricX
    y = metricY
    for ii in paretoData.index:
        w=oapackage.doubleVector( (paretoData[x][ii], paretoData[y][ii]) )
        pareto.addvalue(w, ii)
    for i in pareto.allindices():
        paretoData["front"][i] = True
        
    fig = px.scatter(paretoData, x=x, y=y, color="front")
    fig.show()
    return paretoData

def showPareto2D(df, metricX, metricY):
    paretoData = df.copy()
    paretoData["front"] = False
    pareto=oapackage.ParetoDoubleLong()
    x = metricX + "-perc"
    y = metricY + "-perc"
    for ii in paretoData.index:
        w=oapackage.doubleVector( (paretoData[x][ii], paretoData[y][ii]) )
        pareto.addvalue(w, ii)
    for i in pareto.allindices():
        paretoData["front"][i] = True
        
    fig = px.scatter(paretoData, x=x, y=y, color="front")
    fig.show()
    return paretoData

def showPareto3D(df, metricX, metricY, metricZ):
    paretoData = df.copy()
    paretoData["front"] = False
    pareto=oapackage.ParetoDoubleLong()
    x = metricX + "-perc"
    y = metricY + "-perc"
    z = metricZ + "-perc"
    for ii in paretoData.index:
        w=oapackage.doubleVector( (paretoData[x][ii], paretoData[y][ii], paretoData[z][ii]) )
        pareto.addvalue(w, ii)
    for i in pareto.allindices():
        paretoData["front"][i] = True
        
    fig = px.scatter_3d(paretoData, x=x, y=y, z=z, color="front")
    fig.show()
    return paretoData

def getParetoData(df, scoreMetrics):
    paretoData = df.copy()
    paretoData["front"] = False
    pareto=oapackage.ParetoDoubleLong()
    scoreMetricsPerc = [metric + "-perc" for metric in scoreMetrics]
    for ii in paretoData.index:
        [paretoData[metric][ii] for metric in scoreMetricsPerc]
        w=oapackage.doubleVector( [paretoData[metric][ii] for metric in scoreMetricsPerc] )
        pareto.addvalue(w, ii)
    for i in pareto.allindices():
        paretoData["front"][i] = True
    return paretoData

def showSunburst(df, path, metric):
    fig = px.sunburst(df, path=path, values=metric)
    fig.update_traces(textinfo="label+percent entry")
    fig.update_layout(autosize=True, height = 1000)
    fig.show()

def cellColor(val):
    val = float(val)
    if val > 0.0: color = 'lightgreen'
    elif val < 0.0: color = 'lightcoral'
    else: color = 'khaki'
    return 'background-color: %s' % color

def cellColorCol(column):
    metric = column.name.replace("_%", "")
    if metric in metricsSorting:
        if metricsSorting[metric]:
            #return [badColor if float(val) > 0.0 else goodColor if float(val) < 0.0 else neutralColor for val in column]
            return [goodColor if float(val) > 0.0 else badColor if float(val) < 0.0 else neutralColor for val in column]
        else:
            #return [goodColor if float(val) > 0.0 else badColor if float(val) < 0.0 else neutralColor for val in column]
            return [badColor if float(val) > 0.0 else goodColor if float(val) < 0.0 else neutralColor for val in column]
    return [whiteColor for val in column]

def compareRows(row1, row2):
    compareDfAll = pd.DataFrame([row1, row2])
    compareDf = compareDfAll.copy()
    #for col in compareDf.columns:
    #    if len(compareDf[col].unique()) == 1:
    #        compareDf.drop(col, inplace=True, axis=1)
    compareDf = compareDf.select_dtypes(include=['number'])
    display(compareDf)
    changesDf = compareDf.pct_change()
    changesDf = changesDf.iloc[1] * 100.0
    compareDfAll.loc['%_change'] = compareDfAll.iloc[0]

    for col, value in changesDf.iteritems():
        compareDfAll[col]['%_change'] = value
        
    #with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    #    print(compareDfAll.dtypes)
    #print('executionTimeHostSelectionStDev' in changesDf.index.tolist())
    compareDfAll.loc['%_change'] = compareDfAll.loc['%_change'].fillna(0)
    
    #compareDfAllStyler = compareDfAll.style.applymap(cellColor, subset=(['%_change'], changesDf.index.tolist()))
    compareDfAllStyler = compareDfAll.style.apply(cellColorCol, subset=(['%_change'], changesDf.index.tolist()))
    display(compareDfAllStyler)

def getColoredMetrics(df, metrics):
    lowBestMetrics = []
    highBestMetrics = []
    undefinedMetrics = []
    for metric in metrics:
        if metric in metricsSorting:
            if metricsSorting[metric]:
                lowBestMetrics.append(metric+"_%")
            else:
                highBestMetrics.append(metric+"_%")
        else:
            undefinedMetrics.append(metric+"_%")
    #metrics = [m + "_%" for m in metrics]
    return df.style.background_gradient(cmap='RdYlGn',axis=1, vmin=-100, vmax=100, subset=lowBestMetrics) \
            .background_gradient(cmap='RdYlGn_r',axis=1, vmin=-100, vmax=100, subset=highBestMetrics) \
            .background_gradient(cmap='gray',axis=1, vmin=-100, vmax=100, subset=undefinedMetrics)
    #return df.style.apply(cellColorCol, subset=metrics)

def getExperiment(experiments, characteristics = {}, verbose=False):
    currentCharacteristics = characteristics.copy()
    allCharacteristics = characteristics.copy()
    filtered = getAllowedFilteredDf(experiments, currentCharacteristics)
    while len(filtered.index) > 1:
        showOptions(filtered)
        print("# solutions:", len(filtered.index))
        currentCharacteristics.clear()
        newCharacteristics = input("Select characteristics: ").replace(" ", "").split(",")
        for characteristic in newCharacteristics:
            key, value = characteristic.split(":")
            currentCharacteristics[key] = [value]
        allCharacteristics.update(currentCharacteristics)
        print(currentCharacteristics)
        filtered = getAllowedFilteredDf(filtered, currentCharacteristics)
    if verbose:
        print(allCharacteristics)
    return filtered

def compareReferenceWithOthers(reference, df, metrics, dropAbsoluteMetrics=False):
    percDf = df.copy()
    columnsToRemove = df.columns.tolist().copy()
    for metric in metrics:
        percDf[metric+"_%"] = (1.0 - (percDf[metric] / reference[metric].iloc[0])) * 100
    for col in experimentConfigurations:
        #print(col)
        if col in columnsToRemove: columnsToRemove.remove(col)
    if not dropAbsoluteMetrics:
        for col in metrics:
            if col in columnsToRemove: columnsToRemove.remove(col)
    percDf.drop(columnsToRemove, axis=1, inplace=True)
    return percDf

def compareReferenceWithImprovedOnes(df, referenceFilter, improvedFilters, metrics, dropAbsoluteMetrics=False):
    reference = getExperiment(df, referenceFilter)
    improvedDf = getAllowedFilteredDf(df, improvedFilters)
    return reference, compareReferenceWithOthers(reference, improvedDf, metrics, dropAbsoluteMetrics)

def combinatoryReferenceComparison(df, referenceFilters, improvedFilters, metrics, improvedFiltersOverrideWithReference=[], dropAbsoluteMetrics=False):
    comparisons = []
    paramGrid = ParameterGrid(referenceFilters)
    for referenceFilter in paramGrid:
        for key in referenceFilter:
            referenceFilter[key] = [referenceFilter[key]]
        currentImprovedFilters = improvedFilters.copy()
        for overrideFilter in improvedFiltersOverrideWithReference:
            currentImprovedFilters[overrideFilter] = referenceFilter[overrideFilter]
        reference, comparisonDf = compareReferenceWithImprovedOnes(df, referenceFilter, currentImprovedFilters, metrics, dropAbsoluteMetrics)
        comparisons.append({"reference": reference, "comparison": comparisonDf, "referenceFilter": referenceFilter, "metrics": metrics})
    return comparisons

def getDfWithClearConfigurationColumnsWithSameValue(df, retainColumns=[], hideColumns=[]):
    dfCopy = df.copy()
    for experimentConfiguration in experimentConfigurations:
        if experimentConfiguration in retainColumns:
            continue
        valueCounts = dfCopy[experimentConfiguration].value_counts()
        if len(valueCounts) == 1:
            dfCopy.drop(experimentConfiguration, axis=1, inplace=True)
    for col in hideColumns:
        if col in dfCopy: dfCopy.drop(col, axis=1, inplace=True)
    return dfCopy

def showCombinatoryData(combinations, showDescribe=False, describeRows="all"):
    for combination in combinations:
        print(combination["referenceFilter"])
        display(combination["reference"])
        cleanComparisonDf = getDfWithClearConfigurationColumnsWithSameValue(combination["comparison"], retainColumns=combination["referenceFilter"].keys(), hideColumns=hideColumns)
        print("Reference:")
        display(combination["reference"].filter(cleanComparisonDf.columns))
        #display(cleanComparisonDf)
        colored = getColoredMetrics(cleanComparisonDf, combination["metrics"])
        print("Improvements:")
        display(colored)
        if showDescribe:
            describedDf = cleanComparisonDf.describe()
            if describeRows != "all":
                describedDf = describedDf.loc[describeRows]
            describedColored = getColoredMetrics(describedDf, combination["metrics"])
            print("Describe:")
            display(describedColored)

def getConcatColumn(df, columns):
    return df.apply(lambda row:"_".join(tuple([str(row[col]) for col in columns])), axis=1)

def getConcatColumnShort(df, columns, length=4):
    return df.apply(lambda row:"_".join(tuple([str(row[col][:min(len(str(row[col])), length)]) for col in columns])), axis=1)